# Pipeline Multimodal RGBT de Contagem de Multidão - DEF-rgbtcc

Este notebook executa a estimativa de densidade e contagem de pessoas em multidão utilizando imagens de modalidade dupla (RGB + Termal) via a biblioteca `head_counting` e a arquitetura **`DEF-rgbtcc`** (ArXiv 2509.17079).

A arquitetura conta com:
- **Dual-Stream Synchronized Reader**: Leitura concorrente e sincronizada dos feeds RGB e Termal.
- **DEF-rgbtcc Model Handler**: Suporte nativo a motores TensorRT (`model_fp16.trt`), PyTorch (`model.pth`) e HuggingFace.
- **Density Heatmap Video Writer**: Renderização de mapa de calor de densidade 2D colorido (`JET`, `HOT`, `TURBO`) sobreposto à imagem visível.

---

## 📚 Referências e Créditos de Arquitetura

* **Artigo Científico**: [DEF-rgbtcc: Dual-Modulation Framework for RGB-T Crowd Counting via Spatially Modulated Attention and Adaptive Fusion](https://arxiv.org/abs/2509.17079)
* **Repositório HuggingFace**: [ilessio-aiflowlab/DEF-rgbtcc](https://huggingface.co/ilessio-aiflowlab/DEF-rgbtcc)
* **Ecossistema**: Módulo ANIMA Defense Ecosystem (Wave 8)

### 1. Seleção da Configuração Dual-Stream RGBT

Escolha qual cenário multimodal YAML deseja executar:
- `data_rgbt_day.yaml` (Cenário Diurno Dual-Stream)
- `data_rgbt_night.yaml` (Cenário Noturno Dual-Stream)

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório app ao path para poder importar a biblioteca head_counting
sys.path.append(str(Path("../app").resolve()))

# Escolha a configuração desejada
CONFIG_FILE = "../app/data_rgbt_day.yaml"  # Altere para '../app/data_rgbt_night.yaml' para o cenário noturno

config_path = Path(CONFIG_FILE).resolve()
print(f"Configuração RGBT ativa selecionada: {config_path}")

### 2. Inicialização do Pipeline e Processamento RGBT

Executamos o pipeline multimodal RGBT. O sistema realiza automatizadamente:
1. Validação e inicialização das threads de leitura dual-stream (`video_rgb` + `video_thermal`).
2. Resolução de prioridade de pesos (TensorRT > SafeTensors > PyTorch > HF).
3. Inferência multimodal em lote via `DEFModelHandler`.
4. Renderização do mapa de densidade 2D em vídeo de mapa de calor de alta resolução.
5. Registro estatístico em CSV (`frame_counts.csv`), JSON (`summary.json`) e telemetria MLflow.

In [ ]:
import logging
from head_counting import run_pipeline

# Configura exibição de logs básicos no notebook
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# Executa o pipeline completo
SUMMARY = run_pipeline(config_path)

### 3. Exibição dos Resultados e Mini-Dashboard RGBT

Renderizamos as estatísticas de multidão obtidas pelo processamento e exibimos o vídeo de mapa de calor resultante.

In [ ]:
from IPython.display import Video, display

print("=" * 60)
print("          RESUMO DO PROCESSAMENTO MULTIMODAL RGBT")
print("=" * 60)
print(f"Sumário JSON (Estatísticas): {SUMMARY['outputs']['summary_json']}")
print(f"CSV de Contagem Temporal:    {SUMMARY['outputs']['frame_counts_csv']}")
print(f"Diretório de Snapshots:      {SUMMARY['outputs']['snapshots_dir']}")
print(f"Vídeo Resultante (Heatmap):   {SUMMARY['outputs']['annotated_video']}")
print("=" * 60)

if 'counts' in SUMMARY:
    print("\n[INFO] Estatísticas de Estimativa de Multidão RGBT:")
    for key, value in SUMMARY['counts'].items():
        label = key.replace('_', ' ').title()
        print(f" ├─ {label}: {value}")
    print("\n")

video_path = Path(SUMMARY['outputs']['annotated_video'])
if video_path.exists():
    print("Visualizando Vídeo de Densidade Calorífica RGBT...")
    display(Video(str(video_path), embed=False, width=960))
else:
    print(f"[AVISO] Vídeo anotado não encontrado em: {video_path}")